# Data Cleaning Pipeline
## Neighborhood Characteristics and Restaurant Health Inspection Outcomes in New York City

**Group Members:** Nicole Liu, Aziza Temirova, Tristan Patterson, Yuchun Wu

This notebook obtains the **cleaned dataset from the raw datasets**. All inputs are local CSV files placed in `data_raw/` — no API access or login is required to reproduce the cleaned data from the raw data.

| # | Raw file in `data_raw/` | Source (persistent public link, no login) |
|---|---|---|
| 1 | `DOHMH_New_York_City_Restaurant_Inspection_Results_*.csv` | [DOHMH Restaurant Inspection Results](https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j) → Export → CSV |
| 2 | `311_Service_Requests_from_2020_to_Present_*.csv` | [311 Service Requests 2020–Present](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2020-to-Present/erm2-nwe9) → filter *Created Date* to the analysis window, then Export → CSV (the unfiltered file is ~20 GB) |
| 3 | `ACSDT5Y*.B*-Data.csv` (4 files, optional; any vintage) | [data.census.gov](https://data.census.gov/): tables **B01003**, **B19013**, **B17001**, **B15003**, geography *ZCTA → all ZCTAs fully/partially within New York*, Download → CSV (extract the `-Data.csv` file from each ZIP) |
| 4 | `2023_Gaz_zcta_national.txt` (optional) | [Census Gazetteer ZCTA file](https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2023_Gazetteer/2023_Gaz_zcta_national.zip) — land area for population density |

Files 1 and 2 are read in **chunks**, so the notebook runs even on the full multi-gigabyte exports. If the optional demographic files are not present, the pipeline still completes and simply leaves the demographic columns missing.

**How to run:** `pip install pandas numpy`, place the raw files in `data_raw/`, then *Run All*. To knit the executed HTML: `jupyter nbconvert --to html --execute data_preparation.ipynb`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Machine Learning 2")
os.chdir(PROJECT_DIR)

print("Current folder:", Path.cwd())
print("data_raw exists:", Path("data_raw").exists())


import os

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

# ----------------------------- configuration -----------------------------
START_DATE = "2023-01-01"          # analysis window for inspections and 311 complaints

import glob

def find_raw(patterns, default):
    """Return the newest file matching any of the given patterns, else the default path."""
    for pat in patterns:
        m = sorted(glob.glob(pat))
        if m:
            return m[-1]
    return default

# Raw bulk exports downloaded from the NYC Open Data portal (date-stamped filenames are detected)
RAW_INSPECTIONS_CSV = find_raw(["data_raw/DOHMH*Restaurant*Inspection*.csv"],
                               "data_raw/DOHMH_New_York_City_Restaurant_Inspection_Results.csv")
RAW_311_CSV         = find_raw(["data_raw/311_Service_Requests*.csv"],
                               "data_raw/311_Service_Requests_from_2020_to_Present.csv")

# Optional demographic inputs (the notebook skips them gracefully if absent).
# data.census.gov downloads are named ACSDT5Y<vintage>.<table>-Data.csv; any vintage
# placed in data_raw/ is detected automatically (the most recent one, if several).
def find_acs_file(table):
    matches = sorted(glob.glob(f"data_raw/ACSDT5Y*{table}-Data.csv"))   # tolerant of . or _ separators
    return matches[-1] if matches else f"data_raw/ACSDT5Y2024_{table}-Data.csv"

ACS_TABLES = ["B01003",   # total population
              "B19013",   # median household income
              "B17001",   # poverty status
              "B15003"]   # educational attainment
ACS_FILES = {t: find_acs_file(t) for t in ACS_TABLES}

GAZETTEER_FILE = "data_raw/2023_Gaz_zcta_national.txt"  # ZCTA land area (for density)

NYC_BOROUGHS = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]

# NYC ZIP codes begin with these prefixes (100-104: Manhattan/Bronx/Staten Island; 110-114, 116: Brooklyn/Queens)
NYC_ZIP_PREFIXES = ("100", "101", "102", "103", "104", "110", "111", "112", "113", "114", "116")

def norm_col(c):
    """Normalize a column name: the portal bulk export uses 'INSPECTION DATE', the API uses 'inspection_date'."""
    return c.strip().lower().replace(" ", "_")

DATE_FORMATS = ("%m/%d/%Y %I:%M:%S %p",   # portal 311 export: 07/02/2026 12:34:56 PM
                "%m/%d/%Y",                # portal DOHMH export: 04/24/2024
                "%Y-%m-%dT%H:%M:%S.%f",    # API ISO timestamps
                "%Y-%m-%d")

def parse_dates(s):
    """Vectorized date parsing: try each known export format; fall back to inference only if none fit."""
    for fmt in DATE_FORMATS:
        parsed = pd.to_datetime(s, format=fmt, errors="coerce")
        if parsed.notna().mean() > 0.9:
            return parsed
    return pd.to_datetime(s, errors="coerce")

os.makedirs("data_clean", exist_ok=True)
for f in [RAW_INSPECTIONS_CSV, RAW_311_CSV]:
    print(f"{f}: {'FOUND' if os.path.exists(f) else 'MISSING -- place the raw CSV here'}")

Mounted at /content/drive
Current folder: /content/drive/MyDrive/Machine Learning 2
data_raw exists: True
data_raw/DOHMH_New_York_City_Restaurant_Inspection_Results_20260703.csv: FOUND
data_raw/311_Service_Requests_from_2020_to_Present_20260703.csv: FOUND


## 1. Raw dataset 1 — DOHMH Restaurant Inspection Results

The raw file is **violation-level**: one row per violation cited, so one inspection with three violations appears as three rows sharing the same `camis` (restaurant ID), `inspection_date`, `score`, and `grade`. We read only the columns the project needs, in chunks, keeping rows inside the analysis window with a non-missing score. This also removes the dataset's placeholder rows for not-yet-inspected establishments, which carry an `inspection_date` of `1900-01-01` and no score.

In [ ]:
insp_cols = ["camis", "dba", "boro", "zipcode", "cuisine_description",
             "inspection_date", "inspection_type", "action",
             "violation_code", "critical_flag", "score", "grade"]

insp_col_set = set(insp_cols)
chunks = []
n_read = 0
for chunk in pd.read_csv(RAW_INSPECTIONS_CSV, usecols=lambda c: norm_col(c) in insp_col_set,
                         dtype=str, chunksize=200_000):
    chunk.columns = [norm_col(c) for c in chunk.columns]
    n_read += len(chunk)
    chunk["inspection_date"] = parse_dates(chunk["inspection_date"])
    chunk["score"] = pd.to_numeric(chunk["score"], errors="coerce")
    keep = chunk[(chunk["inspection_date"] >= START_DATE) & chunk["score"].notna()]
    chunks.append(keep)

insp = pd.concat(chunks, ignore_index=True)
print(f"Reading from: {RAW_INSPECTIONS_CSV}")
print(f"Raw violation-level rows read: {n_read:,}")
print(f"Rows in analysis window with a score: {len(insp):,}")
insp.head()

Reading from: data_raw/DOHMH_New_York_City_Restaurant_Inspection_Results_20260703.csv
Raw violation-level rows read: 296,044
Rows in analysis window with a score: 258,567


,camis,dba,boro,zipcode,cuisine_description,inspection_date,action,violation_code,critical_flag,score,grade,inspection_type
0,40732665,KATZ'S DELICATESSEN,Manhattan,10002,Sandwiches,2024-04-24,Violations were cited in the following area(s).,08A,Not Critical,12.0,A,Cycle Inspection / Re-inspection
1,41639191,CHINA TASTES,Brooklyn,11207,Chinese,2023-12-11,Violations were cited in the following area(s).,06F,Critical,29.0,NaN,Cycle Inspection / Initial Inspection
2,50046429,CHARO RESTAURANT,Queens,11368,Spanish,2026-01-23,Violations were cited in the following area(s).,08A,Not Critical,19.0,Z,Cycle Inspection / Re-inspection
3,41692596,SAANG THAI,Queens,11101,Thai,2023-05-23,Violations were cited in the following area(s).,04L,Critical,35.0,NaN,Cycle Inspection / Initial Inspection
4,41020005,UNCLE BILL'S DINER,Queens,11354,American,2026-06-09,Violations were cited in the following area(s).,06F,Critical,7.0,A,Cycle Inspection / Initial Inspection


## 2. Cleaning the inspection data

Steps (matching the Data Preparation section of the proposal):
1. Drop malformed records (missing IDs, negative scores).
2. Standardize ZIP codes to 5-digit strings; keep only valid NYC ZIP codes and the five boroughs.
3. Restrict to **Cycle Inspections** (initial and re-inspection) — the routine unannounced inspections whose scores are comparable across restaurants. Special programs (task force, administrative, pre-permit) follow different rules.
4. Collapse violation-level rows to **inspection level**, creating derived variables `n_violations` and `n_critical_violations`.
5. Keep the **most recent inspection per restaurant**, so each restaurant contributes one observation.
6. Derive `grade_from_score` using the DOHMH cutoffs (A: 0–13, B: 14–27, C: 28+) and an `is_grade_A` indicator — many recent inspections carry a `grade` of `N`/`Z`/`P` (pending), so the score-based grade is more complete than the posted letter grade.

In [ ]:
# --- 1. malformed records ---
n0 = len(insp)
insp = insp.dropna(subset=["camis"])
insp = insp[insp["score"] >= 0]
print(f"Dropped {n0 - len(insp):,} malformed rows. Remaining: {len(insp):,}")

# --- 2. geography ---
insp["zipcode"] = insp["zipcode"].astype(str).str.extract(r"(\d{5})")[0]
n0 = len(insp)
insp = insp[insp["boro"].isin(NYC_BOROUGHS)]
insp = insp[insp["zipcode"].notna() & insp["zipcode"].str.startswith(NYC_ZIP_PREFIXES)]
print(f"Dropped {n0 - len(insp):,} rows with invalid borough or non-NYC ZIP. Remaining: {len(insp):,}")

# --- 3. cycle inspections only ---
n0 = len(insp)
insp = insp[insp["inspection_type"].str.startswith("Cycle Inspection", na=False)]
print(f"Dropped {n0 - len(insp):,} non-cycle inspection rows. Remaining: {len(insp):,}")

# --- 4. violation-level -> inspection-level ---
def n_unique_codes(s):
    return s.dropna().nunique()

insp_level = (
    insp.sort_values(["camis", "inspection_date"])
        .groupby(["camis", "inspection_date"], as_index=False)
        .agg(
            dba=("dba", "first"),
            boro=("boro", "first"),
            zipcode=("zipcode", "first"),
            cuisine=("cuisine_description", "first"),
            inspection_type=("inspection_type", "first"),
            score=("score", "max"),             # score is constant within an inspection
            grade=("grade", "first"),
            n_violations=("violation_code", n_unique_codes),
            n_critical_violations=("critical_flag", lambda s: (s == "Critical").sum()),
        )
)
print(f"Inspection-level rows: {len(insp_level):,}")

# --- 5. most recent inspection per restaurant ---
restaurants = (
    insp_level.sort_values(["camis", "inspection_date"])
              .groupby("camis", as_index=False)
              .tail(1)
              .reset_index(drop=True)
)
print(f"Unique restaurants (most recent cycle inspection each): {len(restaurants):,}")

# --- 6. derived outcome variables ---
def grade_from_score(s):
    if s <= 13:
        return "A"
    elif s <= 27:
        return "B"
    return "C"

restaurants["grade_from_score"] = restaurants["score"].apply(grade_from_score)
restaurants["is_grade_A"] = (restaurants["grade_from_score"] == "A").astype(int)

restaurants.head()

Dropped 0 malformed rows. Remaining: 258,567
Dropped 2,855 rows with invalid borough or non-NYC ZIP. Remaining: 255,712
Dropped 57,161 non-cycle inspection rows. Remaining: 198,551
Inspection-level rows: 59,841
Unique restaurants (most recent cycle inspection each): 21,472


,camis,inspection_date,dba,boro,zipcode,cuisine,inspection_type,score,grade,n_violations,n_critical_violations,grade_from_score,is_grade_A
0,30075445,2026-02-27,MORRIS PARK BAKE SHOP,Bronx,10462,Bakery Products/Desserts,Cycle Inspection / Initial Inspection,7.0,A,2,1,A,1
1,30191841,2025-02-20,D.J. REYNOLDS,Manhattan,10019,Irish,Cycle Inspection / Re-inspection,10.0,A,2,2,A,1
2,40356018,2025-09-17,RIVIERA CATERERS,Brooklyn,11224,American,Cycle Inspection / Initial Inspection,10.0,A,2,1,A,1
3,40356483,2026-06-10,WILKEN'S FINE FOOD,Brooklyn,11234,Sandwiches,Cycle Inspection / Initial Inspection,12.0,A,3,1,A,1
4,40356731,2026-06-25,TASTE THE TROPICS ICE CREAM,Brooklyn,11226,Frozen Desserts,Cycle Inspection / Initial Inspection,5.0,A,1,1,A,1


## 3. Raw dataset 2 — 311 Service Requests (sanitation complaints)

The raw 311 file is read in chunks (only 4 of its ~40 columns), so this cell works even on the full multi-gigabyte export. Within each chunk we keep requests in the analysis window, classify sanitation-related complaints, and aggregate counts by ZIP code — the full raw rows are never held in memory at once.

Complaint type labels changed over the years (e.g., *Dirty Condition* vs *Dirty Conditions*, *UNSANITARY CONDITION*), so sanitation complaints are matched by **keyword** rather than exact names, with rodent complaints tracked separately.

In [ ]:
SANITATION_PATTERN = (
    "RODENT|DIRTY|UNSANITARY|SANITATION CONDITION|MISSED COLLECTION|"
    "ILLEGAL DUMPING|LITTER|SWEEPING|GARBAGE|RESIDENTIAL DISPOSAL|COMMERCIAL DISPOSAL"
)

# Resolve the needed columns against whatever header the export actually has.
# Each target column accepts several known aliases; if none match, the error
# message lists the file's real columns so the alias table can be extended.
COLUMN_ALIASES = {
    "created_date":   ["created_date", "created", "date_created", "created_at"],
    "complaint_type": ["complaint_type", "problem_(formerly_complaint_type)",
                       "problem", "complaint", "complainttype"],
    "incident_zip":   ["incident_zip", "zip", "zip_code", "zipcode", "incident_zip_code"],
}

header = pd.read_csv(RAW_311_CSV, nrows=0)
norm_to_raw = {norm_col(c): c for c in header.columns}
resolved = {}
for want, aliases in COLUMN_ALIASES.items():
    hit = next((a for a in aliases if a in norm_to_raw), None)
    if hit is None:
        raise ValueError(
            f"Could not find a column for '{want}' in {RAW_311_CSV}.\n"
            f"Columns present in the file: {list(header.columns)}\n"
            f"Add the matching name to COLUMN_ALIASES above and re-run."
        )
    resolved[want] = norm_to_raw[hit]
print("Column mapping:", {k: v for k, v in resolved.items()})

agg_parts = []
matched_types = set()
n_read = n_window = 0

for chunk in pd.read_csv(RAW_311_CSV, usecols=list(resolved.values()),
                         dtype=str, chunksize=500_000):
    chunk = chunk.rename(columns={v: k for k, v in resolved.items()})
    n_read += len(chunk)
    chunk["created_date"] = parse_dates(chunk["created_date"])
    chunk = chunk[chunk["created_date"] >= START_DATE]
    n_window += len(chunk)
    if chunk.empty:
        continue
    chunk["zipcode"] = chunk["incident_zip"].astype(str).str.extract(r"(\d{5})")[0]
    chunk = chunk[chunk["zipcode"].notna() & chunk["zipcode"].str.startswith(NYC_ZIP_PREFIXES)]
    ct = chunk["complaint_type"].astype(str).str.upper()
    chunk["is_sanitation"] = ct.str.contains(SANITATION_PATTERN, regex=True)
    chunk["is_rodent"] = ct.str.contains("RODENT")
    matched_types |= set(chunk.loc[chunk["is_sanitation"], "complaint_type"].unique())
    agg_parts.append(
        chunk.groupby("zipcode").agg(
            sanitation_complaints=("is_sanitation", "sum"),
            rodent_complaints=("is_rodent", "sum"),
        )
    )

zip_311 = (
    pd.concat(agg_parts)
      .groupby(level=0).sum()
      .reset_index()
)

if n_read < 100_000:
    print(f"WARNING: only {n_read:,} raw 311 rows -- this looks like a truncated sample, not the full extract.")
    print("Re-export the dataset from the NYC Open Data portal and place the full CSV in data_raw/.\n")
print(f"Reading from: {RAW_311_CSV}")
print(f"Raw 311 rows read: {n_read:,}; rows in analysis window: {n_window:,}")
print(f"ZIP codes with 311 data: {len(zip_311):,}")
print(f"Total sanitation-related complaints: {int(zip_311['sanitation_complaints'].sum()):,}")
print("\nComplaint types classified as sanitation-related:")
for t in sorted(matched_types):
    print("  -", t)
zip_311.head()

Column mapping: {'created_date': 'Created Date', 'complaint_type': 'Problem (formerly Complaint Type)', 'incident_zip': 'Incident Zip'}
Reading from: data_raw/311_Service_Requests_from_2020_to_Present_20260703.csv
Raw 311 rows read: 21,675,313; rows in analysis window: 12,342,447
ZIP codes with 311 data: 247
Total sanitation-related complaints: 1,147,536

Complaint types classified as sanitation-related:
  - Commercial Disposal Complaint
  - Dirty Condition
  - Illegal Dumping
  - Litter Basket Complaint
  - Litter Basket Request
  - Missed Collection
  - Residential Disposal Complaint
  - Rodent
  - Street Sweeping Complaint
  - UNSANITARY CONDITION
  - Unsanitary Animal Facility
  - Unsanitary Animal Pvt Property
  - Unsanitary Condition
  - Unsanitary Pigeon Condition


,zipcode,sanitation_complaints,rodent_complaints
0,10000,10,2
1,10001,3546,246
2,10002,6738,708
3,10003,8291,1047
4,10004,284,55


## 4. Raw dataset 3 — Neighborhood demographics (ACS 5-Year Estimates)

Demographic variables come from four ACS tables downloaded as CSV from data.census.gov at the **ZIP Code Tabulation Area (ZCTA)** level:

| Variable | ACS table | Meaning |
|---|---|---|
| `population` | B01003_001E | Total population |
| `median_hh_income` | B19013_001E | Median household income ($) |
| `poverty_rate` | B17001_002E / B17001_001E | Share of population below the poverty line |
| `pct_bachelors_plus` | B15003_022E–025E / B15003_001E | Share of adults 25+ with a bachelor's degree or higher |
| `pop_density_sqmi` | population / land area | Population per sq. mile (land area from the Census Gazetteer file) |

data.census.gov exports contain a second header row of human-readable labels, which the loader below detects and drops; ZCTA codes are extracted from the `GEO_ID` field (e.g. `860Z200US10001` → `10001`). ZCTAs approximate USPS ZIP codes closely in NYC; we note the approximation as a limitation. If these files are not present, the notebook continues with demographic columns set to missing.

In [ ]:
def load_acs_table(path, columns):
    """Load a data.census.gov table export, returning zipcode + the requested variable columns."""
    df = pd.read_csv(path, dtype=str, low_memory=False, encoding="utf-8-sig")  # utf-8-sig strips the BOM
    # exports include a second header row of descriptive labels -- drop it if present
    if not str(df["GEO_ID"].iloc[0]).upper().startswith(("860", "0400", "0500", "1400")):
        df = df.iloc[1:]
    df["zipcode"] = df["GEO_ID"].astype(str).str.extract(r"US(\d{5})$")[0]
    out = df[["zipcode"] + [c for c in columns if c in df.columns]].copy()
    for c in columns:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
            out.loc[out[c] < 0, c] = np.nan   # ACS uses large negative sentinels for missing
    return out

ACS_VARS = {"B01003": ["B01003_001E"],
            "B19013": ["B19013_001E"],
            "B17001": ["B17001_001E", "B17001_002E"],
            "B15003": ["B15003_001E", "B15003_022E", "B15003_023E", "B15003_024E", "B15003_025E"]}

acs = {}
for table in ACS_TABLES:
    try:
        acs[table] = load_acs_table(ACS_FILES[table], ACS_VARS[table])
        print(f"{table}: loaded {len(acs[table]):,} ZCTAs from {ACS_FILES[table]}")
    except FileNotFoundError:
        print(f"WARNING: {table} data file not found ({ACS_FILES[table]}) -- its variables will be missing.")

demo = pd.DataFrame({"zipcode": pd.Series(dtype=str)})
if "B01003" in acs:
    demo = acs["B01003"].rename(columns={"B01003_001E": "population"})
if "B19013" in acs:
    demo = demo.merge(acs["B19013"].rename(columns={"B19013_001E": "median_hh_income"}),
                      on="zipcode", how="outer")
if "B17001" in acs:
    pov = acs["B17001"]
    pov["poverty_rate"] = pov["B17001_002E"] / pov["B17001_001E"]
    demo = demo.merge(pov[["zipcode", "poverty_rate"]], on="zipcode", how="outer")
if "B15003" in acs:
    edu = acs["B15003"]
    edu["pct_bachelors_plus"] = (edu[["B15003_022E", "B15003_023E", "B15003_024E", "B15003_025E"]].sum(axis=1)
                                 / edu["B15003_001E"])
    demo = demo.merge(edu[["zipcode", "pct_bachelors_plus"]], on="zipcode", how="outer")

for col in ["population", "median_hh_income", "poverty_rate", "pct_bachelors_plus"]:
    if col not in demo.columns:
        demo[col] = np.nan
demo = demo[demo["zipcode"].notna() & demo["zipcode"].str.startswith(NYC_ZIP_PREFIXES)]
print(f"\nNYC-range ZCTAs with demographic data: {len(demo):,}")

# population density from the Census Gazetteer land-area file (optional)
try:
    gaz = pd.read_csv(GAZETTEER_FILE, sep="\t", dtype={"GEOID": str})
    gaz.columns = [c.strip() for c in gaz.columns]
    gaz = gaz[["GEOID", "ALAND_SQMI"]].rename(columns={"GEOID": "zipcode", "ALAND_SQMI": "land_area_sqmi"})
    demo = demo.merge(gaz, on="zipcode", how="left")
    demo["pop_density_sqmi"] = demo["population"] / demo["land_area_sqmi"].replace(0, np.nan)
    demo = demo.drop(columns=["land_area_sqmi"])
    print("Population density computed from Gazetteer land areas.")
except FileNotFoundError:
    print("WARNING: Gazetteer file not found. pop_density_sqmi will be missing.")
    demo["pop_density_sqmi"] = np.nan

demo.head()

B01003: loaded 1,824 ZCTAs from data_raw/ACSDT5Y2024.B01003-Data.csv
B19013: loaded 1,824 ZCTAs from data_raw/ACSDT5Y2024.B19013-Data.csv
B17001: loaded 1,824 ZCTAs from data_raw/ACSDT5Y2024.B17001-Data.csv
B15003: loaded 1,824 ZCTAs from data_raw/ACSDT5Y2024.B15003-Data.csv

NYC-range ZCTAs with demographic data: 223
Population density computed from Gazetteer land areas.


,zipcode,population,median_hh_income,poverty_rate,pct_bachelors_plus,pop_density_sqmi
0,10001,30511.0,129852.0,0.150817,0.726327,48895.833333
1,10002,76873.0,48386.0,0.279341,0.395018,93519.464720
2,10003,53560.0,154262.0,0.126167,0.842855,93636.363636
3,10004,3711.0,NaN,0.035026,0.905097,8138.157895
4,10005,9439.0,190233.0,0.042589,0.858819,129301.369863


## 5. Merging the three sources on ZIP code

The neighborhood measures are left-joined onto the restaurant-level table. A NYC ZIP code absent from the 311 aggregate had zero matched complaints in the window, so 311 counts are filled with 0 after the merge; missing demographic values remain `NaN` (some ZCTAs — airports, parks — have no resident population).

Derived variables (per the proposal): `sanitation_per_1000`, `rodent_per_1000`, and the neighborhood-level `zip_avg_score`.

In [ ]:
df = restaurants.merge(zip_311, on="zipcode", how="left")
df = df.merge(demo, on="zipcode", how="left")

df[["sanitation_complaints", "rodent_complaints"]] = df[["sanitation_complaints", "rodent_complaints"]].fillna(0)

df["sanitation_per_1000"] = 1000 * df["sanitation_complaints"] / df["population"]
df["rodent_per_1000"]     = 1000 * df["rodent_complaints"] / df["population"]
df["zip_avg_score"]       = df.groupby("zipcode")["score"].transform("mean")

# Flag restaurants in non-residential ZIP codes (office towers, transit hubs, airports):
# these ZIPs have no ZCTA or zero resident population, so neighborhood measures are undefined there
df["nonresidential_zip"] = (df["population"].isna() | (df["population"] == 0)).astype(int)

print(f"Final restaurant-level cleaned dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Final restaurant-level cleaned dataset: 21,472 rows x 24 columns


,camis,inspection_date,dba,boro,zipcode,cuisine,inspection_type,score,grade,n_violations,n_critical_violations,grade_from_score,is_grade_A,sanitation_complaints,rodent_complaints,population,median_hh_income,poverty_rate,pct_bachelors_plus,pop_density_sqmi,sanitation_per_1000,rodent_per_1000,zip_avg_score,nonresidential_zip
0,30075445,2026-02-27,MORRIS PARK BAKE SHOP,Bronx,10462,Bakery Products/Desserts,Cycle Inspection / Initial Inspection,7.0,A,2,1,A,1,9429,1067,76320.0,60966.0,0.202424,0.264137,50947.930574,123.545597,13.980608,11.765152,0
1,30191841,2025-02-20,D.J. REYNOLDS,Manhattan,10019,Irish,Cycle Inspection / Re-inspection,10.0,A,2,2,A,1,6086,611,44942.0,128559.0,0.101556,0.782353,68197.268589,135.418984,13.595301,12.234127,0
2,40356018,2025-09-17,RIVIERA CATERERS,Brooklyn,11224,American,Cycle Inspection / Initial Inspection,10.0,A,2,1,A,1,3702,284,50822.0,43648.0,0.254768,0.351531,32767.246937,72.842470,5.588131,14.343750,0
3,40356483,2026-06-10,WILKEN'S FINE FOOD,Brooklyn,11234,Sandwiches,Cycle Inspection / Initial Inspection,12.0,A,3,1,A,1,7527,734,83187.0,97479.0,0.095407,0.394888,11265.845070,90.482888,8.823494,13.746269,0
4,40356731,2026-06-25,TASTE THE TROPICS ICE CREAM,Brooklyn,11226,Frozen Desserts,Cycle Inspection / Initial Inspection,5.0,A,1,1,A,1,23723,2527,95144.0,81014.0,0.159009,0.379395,73019.186493,249.337846,26.559741,12.831325,0


## 6. Validation checks and summary of the cleaned data

In [ ]:
print("=== Missing values per column ===")
print(df.isna().sum().to_string())

print("\n=== Restaurants by borough ===")
print(df["boro"].value_counts().to_string())

print("\n=== Grade distribution (derived from score) ===")
print(df["grade_from_score"].value_counts(normalize=True).round(3).to_string())

print("\n=== Key numeric variables ===")
num_cols = ["score", "n_violations", "n_critical_violations", "sanitation_per_1000",
            "rodent_per_1000", "median_hh_income", "poverty_rate", "pct_bachelors_plus",
            "pop_density_sqmi"]
df[num_cols].describe().round(2)

=== Missing values per column ===
camis                       0
inspection_date             0
dba                         0
boro                        0
zipcode                     0
cuisine                     0
inspection_type             0
score                       0
grade                    2687
n_violations                0
n_critical_violations       0
grade_from_score            0
is_grade_A                  0
sanitation_complaints       0
rodent_complaints           0
population                130
median_hh_income          570
poverty_rate              254
pct_bachelors_plus        254
pop_density_sqmi          130
sanitation_per_1000       154
rodent_per_1000           184
zip_avg_score               0
nonresidential_zip          0

=== Restaurants by borough ===
boro
Manhattan        8399
Brooklyn         5515
Queens           4825
Bronx            1948
Staten Island     785

=== Grade distribution (derived from score) ===
grade_from_score
A    0.781
B    0.154
C    0.065


,score,n_violations,n_critical_violations,sanitation_per_1000,rodent_per_1000,median_hh_income,poverty_rate,pct_bachelors_plus,pop_density_sqmi
count,21472.00,21472.00,21472.00,21318.00,21288.00,20902.00,21218.00,21218.00,21342.00
mean,13.25,2.76,1.42,inf,inf,103652.27,0.16,0.53,52096.46
std,9.34,1.54,1.14,NaN,NaN,40844.58,0.09,0.23,28259.09
min,0.00,0.00,0.00,2.76,0.00,24086.00,0.00,0.00,0.00
25%,8.00,2.00,1.00,96.07,9.04,70603.00,0.10,0.34,31088.48
50%,12.00,2.00,1.00,128.67,13.90,95638.00,0.14,0.52,50034.93
75%,13.00,3.00,2.00,169.21,19.55,137106.00,0.20,0.77,69431.77
max,126.00,15.00,12.00,inf,inf,233636.00,0.89,0.93,172782.61


In [ ]:
# quick sanity check: mean inspection score by borough (preview of research question 1)
df.groupby("boro")["score"].agg(["count", "mean", "median"]).round(2)

,count,mean,median
boro,,,
Bronx,1948,12.87,12.0
Brooklyn,5515,12.88,12.0
Manhattan,8399,12.99,12.0
Queens,4825,14.25,12.0
Staten Island,785,13.56,12.0


## 7. Export the cleaned datasets

In [ ]:
# Restaurant-level cleaned dataset (primary analysis file: one row per restaurant)
out_cols = ["camis", "dba", "boro", "zipcode", "cuisine", "inspection_date", "inspection_type",
            "score", "grade", "grade_from_score", "is_grade_A", "n_violations", "n_critical_violations",
            "sanitation_complaints", "rodent_complaints", "sanitation_per_1000", "rodent_per_1000",
            "population", "pop_density_sqmi", "median_hh_income", "poverty_rate",
            "pct_bachelors_plus", "zip_avg_score", "nonresidential_zip"]
df[out_cols].to_csv("data_clean/cleaned_restaurant_inspections.csv", index=False)

# Neighborhood-level (ZIP) aggregate, for neighborhood-level robustness analyses
zip_level = (
    df.groupby(["zipcode", "boro"], as_index=False)
      .agg(n_restaurants=("camis", "count"),
           avg_score=("score", "mean"),
           pct_grade_A=("is_grade_A", "mean"),
           sanitation_complaints=("sanitation_complaints", "first"),
           rodent_complaints=("rodent_complaints", "first"),
           sanitation_per_1000=("sanitation_per_1000", "first"),
           rodent_per_1000=("rodent_per_1000", "first"),
           population=("population", "first"),
           pop_density_sqmi=("pop_density_sqmi", "first"),
           median_hh_income=("median_hh_income", "first"),
           poverty_rate=("poverty_rate", "first"),
           pct_bachelors_plus=("pct_bachelors_plus", "first"))
)
zip_level.to_csv("data_clean/cleaned_neighborhood_zip_level.csv", index=False)

print("Saved:")
print("  data_clean/cleaned_restaurant_inspections.csv  ", df[out_cols].shape)
print("  data_clean/cleaned_neighborhood_zip_level.csv  ", zip_level.shape)

Saved:
  data_clean/cleaned_restaurant_inspections.csv   (21472, 24)
  data_clean/cleaned_neighborhood_zip_level.csv   (217, 14)


## 8. Summary of data preparation efforts

- **Deduplication and unit of analysis.** The raw inspection file is violation-level; we collapsed it to inspection level (creating `n_violations` and `n_critical_violations` as derived variables) and then kept each restaurant's most recent cycle inspection, so the cleaned dataset has one row per restaurant.
- **Filtering.** Removed placeholder records (`1900-01-01` dates, missing scores), invalid boroughs, non-NYC or malformed ZIP codes, and non-routine inspection types whose scores are not comparable to cycle inspections.
- **Standardization.** ZIP codes normalized to 5-digit strings; dates parsed; scores cast to numeric; ACS negative sentinel values converted to missing.
- **Aggregation.** 311 sanitation-related complaints (matched by keyword to be robust to label changes across years) aggregated to ZIP level in memory-safe chunks, with rodent complaints tracked separately; per-capita rates derived using ACS population.
- **Integration.** All three sources merged on ZIP code (ZCTA for the ACS), producing a restaurant-level analysis file and a ZIP-level aggregate file.
- **Known limitations.** ZCTAs approximate USPS ZIP codes; 311 complaint volume partly reflects reporting propensity, not only true conditions; the most-recent-inspection design means inspection dates vary across restaurants within the window.